In [1]:
# 1 - Imports

import requests
from bs4 import BeautifulSoup
import os
import time
from urllib.parse import urljoin, urlparse

from pathlib import Path

In [2]:
# 2 - Funcoes principais

def baixar_imagens(url, pasta_salvar='imagens_baixadas', max_imagens=10):
    """
    Baixa imagens de uma página web
    
    Args:
        url: URL da página para fazer scraping
        pasta_salvar: Nome da pasta para salvar as imagens
        max_imagens: Número máximo de imagens para baixar
    """
    
    # Criar pasta para salvar as imagens
    if not os.path.exists(pasta_salvar):
        os.makedirs(pasta_salvar)
    
    # Headers para simular um navegador
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        # Fazer requisição à página
        print(f"Acessando: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Verifica se houve erro HTTP
        
        # Parse do HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Encontrar todas as tags de imagem
        imagens = soup.find_all('img')
        print(f"Encontradas {len(imagens)} imagens na página")
        
        contador = 0
        for i, img_tag in enumerate(imagens):
                
            # Obter URL da imagem
            img_url = img_tag.get('src') or img_tag.get('data-src')
            
            if not img_url:
                continue
                
            # Converter URL relativa para absoluta
            img_url = urljoin(url, img_url)
            
            # Validar URL da imagem
            if not img_url.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')):
                # Verificar se a URL contém extensão de imagem
                parsed = urlparse(img_url)
                if not any(ext in parsed.path.lower() for ext in ['.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp']):
                    continue
            
            try:
                # Baixar a imagem
                print(f"Baixando imagem {contador + 1}: {img_url[:80]}...")
                img_response = requests.get(img_url, headers=headers, timeout=10, stream=True)
                img_response.raise_for_status()
                
                # Extrair nome do arquivo da URL
                parsed_url = urlparse(img_url)
                nome_arquivo = os.path.basename(parsed_url.path)
                
                # Se não tiver extensão, adicionar .jpg
                if not nome_arquivo or '.' not in nome_arquivo:
                    nome_arquivo = f"imagem_{i}.jpg"
                
                # Caminho completo para salvar
                caminho_completo = os.path.join(pasta_salvar, nome_arquivo)
                
                # Verificar se arquivo já existe e renomear se necessário
                if os.path.exists(caminho_completo):
                    nome_base, extensao = os.path.splitext(nome_arquivo)
                    caminho_completo = os.path.join(pasta_salvar, f"{nome_base}_{i}{extensao}")
                
                # Salvar a imagem
                with open(caminho_completo, 'wb') as f:
                    for chunk in img_response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                
                print(f"  ✓ Salvo como: {caminho_completo}")
                contador += 1
                
                # Delay para não sobrecarregar o servidor
                time.sleep(0.5)
                
            except Exception as e:
                print(f"  ✗ Erro ao baixar imagem: {e}")
                continue
        
        print(f"\n✅ Download concluído! {contador} imagens salvas em '{pasta_salvar}/'")
        
    except requests.RequestException as e:
        print(f"Erro ao acessar a página: {e}")
    except Exception as e:
        print(f"Erro inesperado: {e}")

# Versão mais avançada com mais opções
def baixar_imagens_avancado(url, **kwargs):
    """
    Versão avançada com mais opções de filtro
    """
    pasta_salvar = kwargs.get('pasta_salvar', 'imagens_baixadas')
    max_imagens = kwargs.get('max_imagens', 20)
    min_largura = kwargs.get('min_largura', 50)  # Filtro por tamanho mínimo
    min_altura = kwargs.get('min_altura', 50)
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        imagens = []
        # Procurar imagens em diferentes tags
        for img in soup.find_all('img'):
            src = img.get('src') or img.get('data-src') or img.get('data-lazy-src')
            if src:
                imagens.append({
                    'url': urljoin(url, src),
                    'alt': img.get('alt', ''),
                    'width': img.get('width'),
                    'height': img.get('height')
                })
        
        print(f"Imagens encontradas: {len(imagens)}")
        
        # Baixar imagens
        for i, img_info in enumerate(imagens[:max_imagens]):
            baixar_imagem_individual(img_info['url'], pasta_salvar, i, headers)
            time.sleep(0.3)
            
    except Exception as e:
        print(f"Erro: {e}")

def baixar_imagem_individual(img_url, pasta, indice, headers):
    """Baixa uma imagem individual"""
    try:
        img_data = requests.get(img_url, headers=headers, timeout=5).content
        
        # Determinar extensão pelo conteúdo
        extensao = determinar_extensao(img_data)
        nome_arquivo = f"imagem_{indice}{extensao}"
        
        with open(os.path.join(pasta, nome_arquivo), 'wb') as f:
            f.write(img_data)
        print(f"Baixada: {nome_arquivo}")
        
    except Exception as e:
        print(f"Falha ao baixar {img_url[:50]}: {e}")

def determinar_extensao(img_data):
    """Tenta determinar a extensão pelo conteúdo da imagem"""
    if img_data.startswith(b'\xff\xd8'):
        return '.jpg'
    elif img_data.startswith(b'\x89PNG'):
        return '.png'
    elif img_data.startswith(b'GIF'):
        return '.gif'
    elif img_data.startswith(b'BM'):
        return '.bmp'
    elif img_data.startswith(b'RIFF') and img_data[8:12] == b'WEBP':
        return '.webp'
    return '.jpg'  # Padrão


In [3]:
# 3 - Funcao auxiliar

def renomear_imagens(caminho_pasta, prefixo="imagem", inicio=0):
    """
    Renomeia todas as imagens em uma pasta para 'imagem (contador)'
    
    Args:
        caminho_pasta: Caminho da pasta contendo as imagens
        prefixo: Prefixo para o nome das imagens (padrão: "imagem")
        iniciar_em: Valor inicial do contador (padrão: 0)
    """
    
    # Converte para objeto Path
    pasta = Path(caminho_pasta)
    
    # Verifica se a pasta existe
    if not pasta.exists():
        print(f"Erro: A pasta '{caminho_pasta}' não existe.")
        return
    
    if not pasta.is_dir():
        print(f"Erro: '{caminho_pasta}' não é uma pasta.")
        return
    
    # Extensões de imagem mais comuns
    extensoes_imagem = {
        '.jpg', '.jpeg', '.png', '.gif', '.bmp', 
        '.tiff', '.tif', '.webp', '.svg', '.ico',
        '.jfif', '.pjpeg', '.pjp', '.avif'
    }
    
    # Obtém lista de arquivos de imagem
    imagens = []
    for arquivo in pasta.iterdir():
        if arquivo.is_file() and arquivo.suffix.lower() in extensoes_imagem:
            imagens.append(arquivo)
    
    # Ordena os arquivos por nome para manter alguma ordem
    imagens.sort(key=lambda x: x.name.lower())
    
    if not imagens:
        print(f"Nenhuma imagem encontrada na pasta '{caminho_pasta}'.")
        return
    
    print(f"Encontradas {len(imagens)} imagens na pasta.")
    print("Renomeando imagens...")
    
    # Renomeia cada imagem
    contador = inicio
    renomeadas = 0
    
    for imagem in imagens:
        # Mantém a extensão original do arquivo
        extensao = imagem.suffix
        
        # Cria o novo nome
        novo_nome = f"{prefixo} ({contador}){extensao}"
        novo_caminho = pasta / novo_nome
        
        # Verifica se o novo nome já existe
        while novo_caminho.exists():
            print(f"Aviso: '{novo_nome}' já existe. Pulando para o próximo número.")
            contador += 1
            novo_nome = f"{prefixo} ({contador}){extensao}"
            novo_caminho = pasta / novo_nome
        
        try:
            # Renomeia o arquivo
            imagem.rename(novo_caminho)
            print(f"Renomeado: {imagem.name} -> {novo_nome}")
            renomeadas += 1
            contador += 1
            
        except Exception as e:
            print(f"Erro ao renomear {imagem.name}: {e}")
    
    print(f"\nConcluído! {renomeadas} imagens foram renomeadas.")


In [4]:
# 4 - Execucao da funcao principal

url_alvo = "https://www.edelkoortsth.com/shop"  # Substitua pela URL desejada
pasta_imagens = 'C:/Users/Kelvin/Desktop/IC/scrapping/imagens'

# Uso simples
baixar_imagens(
    url=url_alvo,
    pasta_salvar=pasta_imagens,
    max_imagens=10
    )
    
print("\n" + "="*50)
print("PARA USAR COM OUTRAS URLs:")
print("1. Substitua 'url_alvo' pelo site desejado")
print("2. Ajuste max_imagens conforme necessário")
print("3. Execute o script")
print("="*50)

Acessando: https://www.edelkoortsth.com/shop
Encontradas 20 imagens na página
Baixando imagem 1: https://static.wixstatic.com/media/76f82d_6e8e80c0afed4488859bb174c9f732ab~mv2.p...
  ✓ Salvo como: C:/Users/Kelvin/Desktop/IC/scrapping/imagens\edelkoortsth_tif.png
Baixando imagem 2: https://static.wixstatic.com/media/0fdef751204647a3bbd7eaa2827ed4f9.png/v1/fill/...
  ✓ Salvo como: C:/Users/Kelvin/Desktop/IC/scrapping/imagens\0fdef751204647a3bbd7eaa2827ed4f9.png
Baixando imagem 3: https://static.wixstatic.com/media/01c3aff52f2a4dffa526d7a9843d46ea.png/v1/fill/...
  ✓ Salvo como: C:/Users/Kelvin/Desktop/IC/scrapping/imagens\01c3aff52f2a4dffa526d7a9843d46ea.png
Baixando imagem 4: https://static.wixstatic.com/media/76f82d_d8e2f4049027449ab492c5597bf63924~mv2.j...
  ✓ Salvo como: C:/Users/Kelvin/Desktop/IC/scrapping/imagens\76f82d_d8e2f4049027449ab492c5597bf63924~mv2.jpg
Baixando imagem 5: https://static.wixstatic.com/media/76f82d_eb51670959474f6084dddfc1148b9fe5~mv2.j...
  ✓ Salvo como: C:/U

In [5]:
# 5 - Execucao da funcao auxiliar
prefixo = 'imagem'
inicio = 0
pasta_imagens = 'C:/Users/Kelvin/Desktop/IC/scrapping/imagens'

renomear_imagens(pasta_imagens, prefixo, inicio)

Encontradas 39 imagens na pasta.
Renomeando imagens...
Renomeado: 00.jpg -> imagem (0).jpg
Renomeado: 1%20(1).jpg -> imagem (1).jpg
Renomeado: 106-107.jpg -> imagem (2).jpg
Renomeado: 28-29.jpg -> imagem (3).jpg
Renomeado: 286-287.jpg -> imagem (4).jpg
Renomeado: 3.jpg -> imagem (5).jpg
Renomeado: 76f82d_13fa5bdb5f26454f8324e58f7aa39088~mv2.avif -> imagem (6).avif
Renomeado: 76f82d_13fa5bdb5f26454f8324e58f7aa39088~mv2.jpg -> imagem (7).jpg
Renomeado: 76f82d_14d992163aaa4dcbb61a84077bf6cd8e~mv2.avif -> imagem (8).avif
Renomeado: 76f82d_1da2ec12b2074f8e89dc28b7408384fe~mv2.jpg -> imagem (9).jpg
Renomeado: 76f82d_1efc168af1044a648a501126c211957c~mv2.avif -> imagem (10).avif
Renomeado: 76f82d_638720acaf7d47eb9755b9b362ce66b2~mv2.avif -> imagem (11).avif
Renomeado: 76f82d_8aedb811555642569161ffc3a4ae1bf8~mv2.avif -> imagem (12).avif
Renomeado: 76f82d_9d79eeae01764543b2669850ad9af6f0~mv2.avif -> imagem (13).avif
Renomeado: 76f82d_aa542b9d195e4116a95c68a9533d4f0d~mv2.avif -> imagem (14).avif
